# Sentence Model with Contrastive Learning out-of-Corpus

This notebook perform evaluation of the sentence model using contrastive learning in the out-of-corpus inference setting.

# Class definition

In [ ]:
import numpy as np
import tensorflow as tf
import pickle
import pandas as pd
import keras
from keras.backend import dropout

def positional_encoding(length, depth):
  depth = depth/2

  positions = np.arange(length)[:, np.newaxis]     # (seq, 1)
  depths = np.arange(depth)[np.newaxis, :]/depth   # (1, depth)

  angle_rates = 1 / (10000**depths)         # (1, depth)
  angle_rads = positions * angle_rates      # (pos, depth)

  pos_encoding = np.concatenate(
      [np.sin(angle_rads), np.cos(angle_rads)],
      axis=-1)

  return tf.cast(pos_encoding, dtype=tf.float32)


class PositionalEmbedding(tf.keras.layers.Layer):
  def __init__(self, d_model):
    super().__init__()
    self.d_model = d_model
    self.pos_encoding = positional_encoding(length=2048, depth=d_model)

  def call(self, x):
    length = tf.shape(x)[1]
    x *= tf.math.sqrt(tf.cast(self.d_model, tf.float32))
    x = x + self.pos_encoding[tf.newaxis, :length, :]
    return x


class MHAttention(tf.keras.layers.Layer):
  def __init__(self, **kwargs):
    super().__init__()
    self.mha = tf.keras.layers.MultiHeadAttention(**kwargs)
    self.layernorm = tf.keras.layers.LayerNormalization()
    self.add = tf.keras.layers.Add()

  def call(self, x):
      attn_output = self.mha(
          query=x,
          value=x,
          key=x)
      x = self.add([x, attn_output])
      x = self.layernorm(x)
      return x


class FeedForward(tf.keras.layers.Layer):
  def __init__(self, d_model, dff, dropout_rate=0.1):
    super().__init__()
    self.seq = tf.keras.Sequential([
      tf.keras.layers.Dense(dff, activation='relu'),
      tf.keras.layers.Dense(d_model),
      tf.keras.layers.Dropout(dropout_rate)
    ])
    self.add = tf.keras.layers.Add()
    self.layer_norm = tf.keras.layers.LayerNormalization()

  def call(self, x):
    x = self.add([x, self.seq(x)])
    x = self.layer_norm(x)
    return x


class EncoderLayer(tf.keras.layers.Layer):
  def __init__(self,*, d_model, num_heads, dff, dropout_rate=0.1):
    super().__init__()

    self.self_attention = MHAttention(
        num_heads=num_heads,
        key_dim=d_model,
        dropout=dropout_rate)

    self.ffn = FeedForward(d_model, dff)

  def call(self, x):
    x = self.self_attention(x)
    x = self.ffn(x)
    return x


class Encoder(tf.keras.layers.Layer):
  def __init__(self, *, num_layers, d_model, num_heads, dff, dropout_rate=0.1):
    super().__init__()

    self.d_model = d_model
    self.num_layers = num_layers

    # self.pos_embedding = PositionalEmbedding(d_model=d_model)

    self.enc_layers = [
        EncoderLayer(d_model=d_model,
                     num_heads=num_heads,
                     dff=dff,
                     dropout_rate=dropout_rate)
        for _ in range(num_layers)]

    self.dropout = tf.keras.layers.Dropout(dropout_rate)

  def call(self, x):
    # `x` is token-IDs shape: (batch, seq_len)
    # x = self.pos_embedding(x)  # Shape `(batch_size, seq_len, d_model)`.

    # Add dropout.
    x = self.dropout(x)

    for i in range(self.num_layers):
      x = self.enc_layers[i](x)

    return x  # Shape `(batch_size, seq_len, d_model)`.


class CrossAttention(tf.keras.layers.Layer):
  def __init__(self, **kwargs):
    super().__init__()
    self.mha = tf.keras.layers.MultiHeadAttention(**kwargs)
    self.layernorm = tf.keras.layers.LayerNormalization()
    self.add = tf.keras.layers.Add()

  def call(self, x, y):
      attn_output = self.mha(
          query=x,
          value=y,
          key=y)
      x = self.add([x, attn_output])
      x = self.layernorm(x)
      return x


class DecoderLayer(tf.keras.layers.Layer):
  def __init__(self,
               *,
               d_model,
               num_heads,
               dff,
               dropout_rate=0.1):
    super(DecoderLayer, self).__init__()

    self.self_attention = MHAttention(
        num_heads=num_heads,
        key_dim=d_model,
        dropout=dropout_rate)

    self.cross_attention = CrossAttention(
        num_heads=num_heads,
        key_dim=d_model,
        dropout=dropout_rate)

    self.ffn = FeedForward(d_model, dff)

  def call(self, x, y):
    x = self.self_attention(x=x)
    x = self.cross_attention(x=x, y=y)

    x = self.ffn(x)  # Shape `(batch_size, seq_len, d_model)`.
    return x


class Decoder(tf.keras.layers.Layer):
  def __init__(self, *, num_layers, d_model, num_heads, dff, dropout_rate=0.1):
    super(Decoder, self).__init__()

    self.d_model = d_model
    self.num_layers = num_layers

    # self.pos_embedding = PositionalEmbedding(d_model=d_model)
    self.dropout = tf.keras.layers.Dropout(dropout_rate)
    self.dec_layers = [
        DecoderLayer(d_model=d_model, num_heads=num_heads,
                     dff=dff, dropout_rate=dropout_rate)
        for _ in range(num_layers)]

  def call(self, x, y):
    # `x` is token-IDs shape (batch, target_seq_len)
    # x = self.pos_embedding(x)  # (batch_size, target_seq_len, d_model)

    x = self.dropout(x)

    for i in range(self.num_layers):
      x  = self.dec_layers[i](x, y)

    # The shape of x is (batch_size, target_seq_len, d_model).
    return x


class MetricLayer(tf.keras.layers.Layer):
  def __init__(self, *, num_layers, d_model, num_heads, dff, num_denses, dropout_rate=0.1):
    super().__init__()
    self.encoder = Encoder(num_layers=num_layers, d_model=d_model,
                           num_heads=num_heads, dff=dff,
                           dropout_rate=dropout_rate)

    self.decoder = Decoder(num_layers=num_layers, d_model=d_model,
                           num_heads=num_heads, dff=dff,
                           dropout_rate=dropout_rate)

    self.denses = [FeedForward(d_model, dff, dropout_rate) for _ in range(num_denses)]
    self.final_layer = tf.keras.layers.Dense(1, activation='sigmoid')
    self.dropout_rate = dropout_rate
    self.dff = dff
    self.d_model = d_model

  def call(self, x, y):
    y = self.encoder(y)  # (batch_size, context_len, d_model)
    x = self.decoder(x, y)  # (batch_size, target_len, d_model)
    for dense in self.denses: #hidden denses
      x = dense(x)
      x = tf.keras.layers.Dropout(self.dropout_rate)(x)
    # x = tf.keras.layers.Flatten()(x) #flatten
    x = tf.math.reduce_sum(x, axis=1)
    x = self.final_layer(x)

    return x

class DistanceLayer(tf.keras.layers.Layer):
  def __init__(self, metric_layer, **kwargs):
      super().__init__(**kwargs)
      self.metric_layer = metric_layer

  def call(self, input1, input2):
      distance = self.metric_layer(input1, input2)
      return tf.reshape(distance, (-1,1))


from sklearn.metrics import f1_score

def f1(x,y,thres):
    yx = np.zeros(x.shape[0])
    yy = np.ones(y.shape[0])
    yhx = np.zeros(x.shape[0])
    yhx[x > thres] = 1
    yhy = np.zeros(y.shape[0])
    yhy[y > thres] = 1
    return f1_score(np.concatenate([yx,yy]), np.concatenate([yhx,yhy]))

# Function to build and evaluate model

In [ ]:
def run_model(answers, gpt1, gpt2, test_answers, test_gpt1, test_gpt2, learning_rate=1e-5, epochs=50, batch_size=256, seeds=(1111,2222,3333,4444,5555,6666,7777,8888,9999,1010)):
  valid_accuracies = []
  valid_f1s = []
  test_accuracies = []
  test_f1s = []

  for seed in seeds:
    np.random.seed(seed)

    #train-test-validation split
    sampling_indices = np.random.uniform(0,1,answers.shape[0])
    train_answers = answers[sampling_indices < 0.9]
    train_gpt1 = gpt1[sampling_indices < 0.9]
    train_gpt2 = gpt2[sampling_indices < 0.9]
    trainX = [np.vstack([train_answers, train_gpt2]), np.vstack([train_gpt1, train_gpt1])]
    trainY = np.vstack([np.ones([train_answers.shape[0], 1]), np.zeros([train_answers.shape[0], 1])])

    valid_answers = answers[sampling_indices >= 0.9]
    valid_gpt1 = gpt1[sampling_indices >= 0.9]
    valid_gpt2 = gpt2[sampling_indices >= 0.9]
    validX = [np.vstack([valid_answers, valid_gpt2]), np.vstack([valid_gpt1, valid_gpt1])]
    validY = np.vstack([np.ones([valid_answers.shape[0], 1]), np.zeros([valid_answers.shape[0], 1])])

    #build model
    emb_shape = 768
    metric_layer = MetricLayer(num_layers=1,
                              d_model=768,
                              num_heads=4,
                              dff=768,
                              num_denses=1)

    input_1 = tf.keras.layers.Input(name="anchor", shape=[pad_length,emb_shape])
    input_2 = tf.keras.layers.Input(name="positive", shape=[pad_length,emb_shape])
    distances = DistanceLayer(metric_layer)(input_1, input_2)
    distance_network = tf.keras.Model(
        inputs=[input_1, input_2], outputs=distances
    )

    #training
    distance_network.compile(
        optimizer=tf.keras.optimizers.Adam(learning_rate),
        loss=tf.keras.losses.MeanSquaredError(),
        metrics=[tf.keras.metrics.MeanSquaredError()],
    )
    distance_network.fit(x=trainX, y=trainY, batch_size=batch_size, epochs=epochs, validation_data=[validX, validY])
    distance_network.compile(
        optimizer=tf.keras.optimizers.Adam(learning_rate/10),
        loss=tf.keras.losses.MeanSquaredError(),
        metrics=[tf.keras.metrics.MeanSquaredError()],
    )
    distance_network.fit(x=trainX, y=trainY, batch_size=batch_size, epochs=10, validation_data=[validX, validY])

    #valid accuracy
    valid_gpt_distances = distance_network.predict([valid_gpt2, valid_gpt1], batch_size=1024)
    valid_answer_distances = distance_network.predict([valid_answers, valid_gpt1], batch_size=1024)
    valid_accuracies.append((valid_gpt_distances < valid_answer_distances).mean())

    #tune threshold
    thresholds = []
    perf = []
    for thres in np.arange(0,2,0.01):
      thresholds.append(thres)
      perf.append(f1(valid_gpt_distances.flatten(), valid_answer_distances.flatten(), thres))
    triplet_thres = thresholds[perf.index(max(perf))]

    #valid f1
    valid_f1s.append(max(perf))

    #test accuracy
    test_gpt_distances = distance_network.predict([test_gpt2, test_gpt1], batch_size=1024)
    test_answer_distances = distance_network.predict([test_answers, test_gpt1], batch_size=1024)
    test_accuracies.append((test_gpt_distances < test_answer_distances).mean())

    #test f1
    test_f1s.append(f1(test_gpt_distances.flatten(), test_answer_distances.flatten(), triplet_thres))
    print(valid_accuracies, test_accuracies, valid_f1s, test_f1s)
    #clear memory
    keras.backend.clear_session()
    print(valid_accuracies, test_accuracies, valid_f1s, test_f1s)
  return valid_accuracies, test_accuracies, valid_f1s, test_f1s

# Run Model

In [ ]:
train_data = 'NQ_GPT4'
test_data = 'SQUAD_GPT4'
pad_length = 5

with open(f'Data/{train_data}_answer_embs.obj', 'rb') as f:
  answers = pickle.load(f)
  answers = tf.keras.preprocessing.sequence.pad_sequences(answers, maxlen=pad_length, padding='post', dtype='float32')
with open(f'Data/{train_data}_gpt1_embs.obj', 'rb') as f:
  gpt1 = pickle.load(f)
  gpt1 = tf.keras.preprocessing.sequence.pad_sequences(gpt1, maxlen=pad_length, padding='post', dtype='float32')
with open(f'Data/{train_data}_gpt2_embs.obj', 'rb') as f:
  gpt2 = pickle.load(f)
  gpt2 = tf.keras.preprocessing.sequence.pad_sequences(gpt2, maxlen=pad_length, padding='post', dtype='float32')
    
with open(f'Data/{test_data}_answer_embs.obj', 'rb') as f:
  test_answers = pickle.load(f)
  test_answers = tf.keras.preprocessing.sequence.pad_sequences(test_answers, maxlen=pad_length, padding='post', dtype='float32')
with open(f'Data/{test_data}_gpt1_embs.obj', 'rb') as f:
  test_gpt1 = pickle.load(f)
  test_gpt1 = tf.keras.preprocessing.sequence.pad_sequences(test_gpt1, maxlen=pad_length, padding='post', dtype='float32')
with open(f'Data/{test_data}_gpt2_embs.obj', 'rb') as f:
  test_gpt2 = pickle.load(f)
  test_gpt2 = tf.keras.preprocessing.sequence.pad_sequences(test_gpt2, maxlen=pad_length, padding='post', dtype='float32')

In [ ]:
valid_accuracies, test_accuracies, valid_f1s, test_f1s = run_model(answers, gpt1, gpt2, test_answers, test_gpt1, test_gpt2, learning_rate=5e-5, batch_size=1024, epochs=30, seeds=[1010])

In [ ]:
np.mean(valid_accuracies), np.mean(test_accuracies), np.mean(valid_f1s), np.mean(test_f1s)